In [1]:
import os
import mlflow
import mlflow.sklearn
import pandas as pd
import joblib
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report

project_root = os.path.abspath('..')
os.chdir(project_root)

mlflow.set_tracking_uri(f'sqlite:///{project_root}/mlflow.db')

df = pd.read_csv('data/processed/combined_en.csv')
X_train, X_test, y_train, y_test = train_test_split(
    df['text'], df['label'],
    test_size=0.2, random_state=42, stratify=df['label']
)

print(f'MLflow version: {mlflow.__version__}')
print(f'Data loaded: {df.shape}')

MLflow version: 3.10.1
Data loaded: (4663, 2)


In [2]:
mlflow.set_experiment('manipulation-detector')

with mlflow.start_run(run_name='tfidf_logreg_baseling'):
    params = {
        'model_type': 'LogisticRegression',
        'max_features': 10_000,
        'ngram_range': '(1,2)',
        'max_iter': 1000,
        'class_weight': 'balanced',
        'test_size': 0.2,
    }
    mlflow.log_params(params)

    pipeline = Pipeline([
        ('tfidf', TfidfVectorizer(max_features=10_000, ngram_range=(1,2), sublinear_tf=True)),
        ('clf', LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42))
    ])

    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)

    f1 = f1_score(y_test, y_pred, average='macro')
    mlflow.log_metric('f1_macro', f1, step=1)
    mlflow.log_metric('test_size', len(X_test))

    mlflow.sklearn.log_model(pipeline, artifact_path='model')

    print(f'F1-macro: {f1:.4f}')
    print(classification_report(y_test, y_pred))

2026/03/15 18:44:47 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/03/15 18:44:47 INFO mlflow.store.db.utils: Updating database tables
2026/03/15 18:44:50 INFO mlflow.tracking.fluent: Experiment with name 'manipulation-detector' does not exist. Creating a new experiment.
2026/03/15 18:44:52 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/15 18:44:52 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


F1-macro: 0.6732
                        precision    recall  f1-score   support

      authority_appeal       0.78      0.65      0.71        94
       demagogy_tricks       0.78      0.53      0.63        34
emotional_manipulation       0.50      0.56      0.53       104
           fear_appeal       0.64      0.67      0.65       140
     rational_argument       0.84      0.85      0.84       561

              accuracy                           0.76       933
             macro avg       0.71      0.65      0.67       933
          weighted avg       0.76      0.76      0.76       933



In [3]:
with mlflow.start_run(run_name='distilbert_finetuned'):
    mlflow.log_params({
        'model_type': 'DistilBERT',
        'epochs': 6,
        'batch_size': 16,
        'lr': 2e-5,
        'max_len': 128,
        'class_weight': 'none',
        'test_size': 0.2
    })

    epoch_metrics = [
        (1, 1.0739, 0.6343, 0.4334),
        (2, 0.7000, 0.7834, 0.5888),
        (3, 0.4469, 0.8649, 0.6929),
        (4, 0.2966, 0.9115, 0.7330),
        (5, 0.1925, 0.9432, 0.7530),
        (6, 0.1369, 0.9619, 0.7708),
    ]

    for epoch, loss, train_acc, f1 in epoch_metrics:
        mlflow.log_metric('loss', loss, step=epoch)
        mlflow.log_metric('train_acc', train_acc, step=epoch)
        mlflow.log_metric('f1_macro', f1, step=epoch)

    print(f'Best F1: 0.7708')

Best F1: 0.7708
